# Week 1 — Information is surprise

**Compression Is Intelligence · WiDS 6.0**

This week you will measure how much information a text *really* contains, and build your first compressor (Huffman coding).

By the end of this notebook you will have:
1. A `surprise()` and `entropy()` function
2. Entropy measurements for four very different texts
3. gzip / bzip2 / xz baselines in **bits per character (bpc)** — the number we will try to beat all month
4. A working Huffman encoder + decoder with a bit-exact round trip

Fill in every cell marked `# TODO`. Each part ends with tests (`assert`s); if they pass silently, you're good.

In [ ]:
import math, os, random, zlib, bz2, lzma, heapq, inspect, json, urllib.request
from collections import Counter
from pathlib import Path

DATA = Path("data")
DATA.mkdir(exist_ok=True)

def load_english(n_chars=200_000):
    p = DATA / "english.txt"
    if not p.exists():
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        urllib.request.urlretrieve(url, p)
    return p.read_text(encoding="utf-8")[:n_chars]

english = load_english()                                                    # Shakespeare (public domain)
code    = inspect.getsource(json.decoder) + inspect.getsource(json.encoder)  # real Python source
alphabet = "abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 ."
noise   = "".join(random.Random(0).choices(alphabet, k=100_000))           # 64 symbols, uniformly random

texts = {"english": english, "python_code": code, "random_64": noise}

# Optional but recommended: paste ~50 KB of Hindi or Marathi text (a news article, Wikipedia page)
# into data/indic.txt and re-run this cell.
indic = DATA / "indic.txt"
if indic.exists():
    texts["hindi_marathi"] = indic.read_text(encoding="utf-8")
else:
    print("Tip: add data/indic.txt with Hindi/Marathi text to compare languages.")

for name, t in texts.items():
    print(f"{name:14s} {len(t):>8,d} chars   sample: {t[:60]!r}")

## Part 1 — Surprise

An event with probability `p` carries **−log₂ p bits** of information. A coin flip (p = 0.5) is 1 bit. Something with p = 1/1024 is 10 bits. Something certain (p = 1) is 0 bits: you learned nothing.

In [ ]:
def surprise(p):
    """Information content of an event with probability p, in bits."""
    # TODO: one line
    raise NotImplementedError

assert surprise(0.5) == 1
assert abs(surprise(1/1024) - 10) < 1e-12
assert surprise(1) == 0
print("Part 1 passed")

## Part 2 — Entropy = average surprise

If characters appear with probabilities p₁, p₂, …, the **entropy** is the average surprise:

H = Σ pᵢ · (−log₂ pᵢ)

Shannon proved no lossless code can use fewer than H bits per symbol *on average* — for a model that treats each character independently (an "order-0" model). Hold on to that last condition; gzip is about to break it.

In [ ]:
def char_probs(text):
    """Return {char: probability} from character counts."""
    # TODO: use Counter
    raise NotImplementedError

def entropy(probs):
    """Entropy in bits of a {symbol: probability} dict."""
    # TODO: use surprise()
    raise NotImplementedError

assert entropy({"a": 0.5, "b": 0.5}) == 1
assert entropy({"a": 1.0}) == 0
assert abs(entropy(char_probs(noise)) - 6) < 0.01   # 64 equally likely symbols -> log2(64) = 6 bits
for name, t in texts.items():
    print(f"{name:14s} order-0 entropy = {entropy(char_probs(t)):.3f} bits/char")
print("Part 2 passed")

## Part 3 — The baselines we will beat

Python ships with three classic compressors. We measure them in **bits per character**: compressed size in bits ÷ number of characters. This is the scoreboard for the whole project.

In [ ]:
def bpc(compress, text):
    """Bits per character achieved by a bytes->bytes compressor on `text`."""
    return 8 * len(compress(text.encode("utf-8"))) / len(text)

compressors = {
    "gzip (zlib -9)": lambda b: zlib.compress(b, 9),
    "bzip2":          lambda b: bz2.compress(b, 9),
    "xz (lzma)":      lambda b: lzma.compress(b, preset=9),
}

print(f"{'text':14s} {'order-0 H':>10s}" + "".join(f"{k:>16s}" for k in compressors))
for name, t in texts.items():
    row = f"{name:14s} {entropy(char_probs(t)):>10.3f}"
    row += "".join(f"{bpc(f, t):>16.3f}" for f in compressors.values())
    print(row)

**Stop and think (answer in your write-up):**
- gzip beats the order-0 entropy on English and code. How is that possible if entropy is a lower bound? *(Hint: re-read the condition in Part 2.)*
- Why can't any compressor beat ~6 bpc on `random_64`?
- Which is more predictable, English or Python code? Why?

## Part 4 — Shannon's guessing game (do this with a friend)

In 1951 Shannon had people guess English text one letter at a time. Count how many guesses each character takes. If a human usually gets it on the first try, English carries very little information per character.

Run `play_shannon_game()` below, hand the keyboard to a friend, and let them guess. The estimate it prints is the entropy of the "number of guesses" distribution, an upper bound on how many bits per character your friend's brain needs.

In [ ]:
def play_shannon_game(text=english, n=60, seed=None):
    rng = random.Random(seed)
    start = rng.randrange(len(text) - n - 200)
    context, target = text[start:start + 200], text[start + 200:start + 200 + n]
    print("Context:\n" + context + "\n" + "-" * 40)
    guesses = []
    for ch in target:
        k = 0
        while True:
            k += 1
            g = input(f"guess next char (try {k}): ")[:1] or " "
            if g == ch:
                break
        guesses.append(k)
        print(f"  correct: {ch!r} in {k} guess(es)")
    est = entropy(char_probs(guesses))
    print(f"\nAverage guesses: {sum(guesses)/len(guesses):.2f}   Estimated bits/char: {est:.2f}")
    return guesses

# play_shannon_game()   # <- uncomment and play. Record the result in your write-up.

## Part 5 — Huffman coding: your first compressor

Huffman gives frequent characters short bit-codes and rare ones long codes, and no code is a prefix of another, so decoding is unambiguous.

Algorithm:
1. Put every symbol in a min-heap keyed by probability.
2. Repeatedly pop the two least likely nodes, merge them into one node whose probability is their sum.
3. When one node remains, walk the tree: left edge = `0`, right edge = `1`.

Tip: heap entries can be tuples `(prob, tiebreak_counter, node)` so Python never has to compare two nodes.

In [ ]:
def huffman_codes(probs):
    """Return {symbol: bitstring} for a {symbol: probability} dict."""
    # TODO: build the tree with heapq, then assign codes
    raise NotImplementedError

def encode(text, codes):
    # TODO: return one long string of '0'/'1'
    raise NotImplementedError

def decode(bits, codes):
    # TODO: walk the bits and emit a symbol whenever the buffer matches a code
    raise NotImplementedError

In [ ]:
# Tests
codes = huffman_codes(char_probs(english))
words = sorted(codes.values())
assert all(not b.startswith(a) for a, b in zip(words, words[1:])), "codes are not prefix-free"

sample = english[:20_000]
assert decode(encode(sample, codes), codes) == sample, "round trip failed"

H = entropy(char_probs(english))
avg = len(encode(english, codes)) / len(english)
assert H <= avg < H + 1, "Huffman should land in [H, H+1)"
print(f"Entropy {H:.3f} bpc   Huffman {avg:.3f} bpc   gzip {bpc(compressors['gzip (zlib -9)'], english):.3f} bpc")
print("Part 5 passed")

## Part 6 — Wrap-up

Make one table with a row per text and these columns: order-0 entropy, Huffman bpc, gzip bpc, bzip2 bpc, xz bpc. Then answer in your README (a few sentences each):

1. Your ranking of the texts by predictability, and why.
2. Why Huffman can never go *below* the order-0 entropy, but gzip can.
3. Huffman always spends at least 1 bit per character. When does that waste the most? *(Think of a text that is 99% the letter `a`.)*
4. Your Shannon-game estimate vs gzip on English. What does the gap tell you about what a better model could achieve?

**Submit:** push this notebook (with outputs) and your README to your GitHub repo by Sunday night.

Next week: arithmetic coding, which fixes problem 3, and context models, which let *us* exploit what gzip exploits.